In [1]:
# https://data.humdata.org/dataset/kontur-population-dataset

In [2]:
import os
os.environ['USE_PYGEOS'] = '0'

import numpy
import pandas
from datetime import datetime, timedelta
import pytz
import geopandas
import dask_geopandas
import shapely
from shapely import wkb, wkt
import pyproj

import subprocess
import shutil
from glob import glob
import json

from functools import partial
from multiprocessing import Pool
from random import shuffle

import matplotlib.pyplot as plt

In [3]:
raw_geoparquet_path = '/data/vector/kontur_population/raw'
processed_geoparquet_path = '/data/vector/kontur_population/processed'
files = glob(os.path.join(raw_geoparquet_path, '*.gpkg'))
len(files)
files

['/data/vector/kontur_population/raw\\kontur_population_20231101.gpkg',
 '/data/vector/kontur_population/raw\\kontur_population_20231101_r4.gpkg',
 '/data/vector/kontur_population/raw\\kontur_population_20231101_r6.gpkg']

In [4]:
%%time
chunk = 1e6
for file in files:
    print(file)
    file = file.replace('\\', '/')
    i=0
    running = True
    while running:
        start = i*chunk
        end = (i+1)*chunk
        print(slice(start, end))

        out_file = os.path.join(
            processed_geoparquet_path,
            os.path.splitext(os.path.split(file)[1])[0] + '_' + str(start) + '_' + str(end) + '.parquet',
        ).replace('\\', '/')

        if not os.path.exists(out_file):
            gdf = geopandas.read_file(file, rows=slice(start, end))
            
            if len(gdf)==0:
                # Reached past the end of the file
                running = False
            else:
                gdf = gdf.to_crs(4326)
        
                if os.path.splitext(file)[0].endswith('_r4'):
                    resolution = '22km'
                elif os.path.splitext(file)[0].endswith('_r6'):
                    resolution = '3km'
                else:
                    resolution = '400m'
                gdf['resolution'] = resolution
        
                gdf['time'] = datetime(2023, 11, 1)
        
                gdf.to_parquet(out_file)

        i+=1


/data/vector/kontur_population/raw\kontur_population_20231101.gpkg
slice(0.0, 1000000.0, None)
slice(1000000.0, 2000000.0, None)
slice(2000000.0, 3000000.0, None)
slice(3000000.0, 4000000.0, None)
slice(4000000.0, 5000000.0, None)
slice(5000000.0, 6000000.0, None)
slice(6000000.0, 7000000.0, None)
slice(7000000.0, 8000000.0, None)
slice(8000000.0, 9000000.0, None)
slice(9000000.0, 10000000.0, None)
slice(10000000.0, 11000000.0, None)
slice(11000000.0, 12000000.0, None)
slice(12000000.0, 13000000.0, None)
slice(13000000.0, 14000000.0, None)
slice(14000000.0, 15000000.0, None)
slice(15000000.0, 16000000.0, None)
slice(16000000.0, 17000000.0, None)
slice(17000000.0, 18000000.0, None)
slice(18000000.0, 19000000.0, None)
slice(19000000.0, 20000000.0, None)
slice(20000000.0, 21000000.0, None)
slice(21000000.0, 22000000.0, None)
slice(22000000.0, 23000000.0, None)
slice(23000000.0, 24000000.0, None)
slice(24000000.0, 25000000.0, None)
slice(25000000.0, 26000000.0, None)
slice(26000000.0, 2700

In [5]:
dask_geopandas.read_parquet(processed_geoparquet_path, split_row_groups=False)

,h3,population,geometry,resolution,time
npartitions=37,,,,,
,object,float64,geometry,object,datetime64[us]
,...,...,...,...,...
...,...,...,...,...,...
,...,...,...,...,...
,...,...,...,...,...
